This is an adaptation of [fast_phoenix_transformerpayne.ipynb](fast_phoenix_transformerpayne.ipynb) to try simpler network formulations.

In [1]:
import timeit
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np

from matplotlib import pyplot as plt
from IPython import display

from astropy import units as u
from astropy.io import fits   

from astropy import visualization as viz
viz.quantity_support()

from jwst import datamodels

from tqdm.auto import tqdm

import jax
import jax.numpy as jnp
import equinox as eqx
import optax

jax.devices()

[CudaDevice(id=0)]

# Timing tests for MLP architectures

The goal is ~1ms per shot, what are the largest mlps that can do this on a sequence size of ~8900?

In [ ]:
rkey = jax.random.key(42)

mlp = eqx.nn.MLP(3, 8900, 300, 3, activation=jax.nn.gelu, key=rkey)

inpt = jnp.ones((1024, 3))

jitted = jax.jit(jax.vmap(mlp))
%time jitted(inpt).block_until_ready()
%timeit jitted(inpt).block_until_ready()

E0901 12:56:43.409768    7032 xtile_compiler.cc:399] Fusion: gemm_fusion_dot.3 = f32[1024,8900]{1,0} fusion(mul.23, constant.23), kind=kCustom, calls=gemm_fusion_dot.3_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","128"]}],"num_ctas":1,"num_stages":3,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0901 12:56:43.409840    7032 xtile_compiler.cc:401] Computation: gemm_fusion_dot.3_computation.clone {
  parameter_0.3 = f32[1024,300]{1,0} parameter(0)
  parameter_1.3 = f32[8900,300]{1,0} parameter(1)
  ROOT dot.7 = f32[1024,8900]{1,0} dot(parameter_0.3, parameter_1.3), lhs_contracting_dims={1}, rhs_contracting_dims={1}, backend_config={"sizes":["64"]}
}
E0901 12:56:43.421641    7050 xtile_compiler.cc:399]

CPU times: user 3.34 s, sys: 165 ms, total: 3.5 s
Wall time: 660 ms


In [ ]:
def mlp_timing(nin, **kwargs):
    if 'key' not in kwargs:
        kwargs['key'] = jax.random.key(42)
    mlp = eqx.nn.MLP(3, 8900, **kwargs)

    nparams = sum(x.size for x in jax.tree.leaves(mlp) if eqx.is_array(x))

    inpt = jnp.ones((nin, 3))

    jitted = jax.jit(jax.vmap(mlp))

    # compile
    jitted(inpt).block_until_ready()

    timer = timeit.Timer('jitted(inpt).block_until_ready()', globals=locals())
    number, _ = timer.autorange()
    timesper = timer.repeat(9, number)

    # ms since the target is 1 ms
    return np.mean(timesper) *1e3  / number, np.std(timesper) *1e3 / number, nparams

res = mlp_timing(1, width_size=300, depth=3, activation=jax.nn.gelu)
print(f"{res[0]:.6f} +/- {res[1]:.6f} ms")

0.049024 +/- 0.005363 ms


In [ ]:
widths = [10, 500, 1000, 3000, 5000]
depths = [2, 3, 4]
act = jax.nn.gelu

for w in widths:
    for d in depths:
        res = mlp_timing(1, width_size=w, depth=d, activation=act)
        print(f"Width: {w}, Depth: {d} -> {res[2] * 1e-6:.3f} Mparams, {res[0]:.6f} +/- {res[1]:.6f} ms")

Width: 10, Depth: 2 -> 0.098 Mparams, 0.056788 +/- 0.011529 ms
Width: 10, Depth: 3 -> 0.098 Mparams, 0.058566 +/- 0.005059 ms
Width: 10, Depth: 4 -> 0.098 Mparams, 0.050507 +/- 0.003069 ms
Width: 500, Depth: 2 -> 4.711 Mparams, 0.049296 +/- 0.004406 ms
Width: 500, Depth: 3 -> 4.962 Mparams, 0.074485 +/- 0.028146 ms
Width: 500, Depth: 4 -> 5.212 Mparams, 0.052171 +/- 0.004400 ms
Width: 1000, Depth: 2 -> 9.914 Mparams, 0.061714 +/- 0.009086 ms
Width: 1000, Depth: 3 -> 10.915 Mparams, 0.074863 +/- 0.012933 ms
Width: 1000, Depth: 4 -> 11.916 Mparams, 0.092181 +/- 0.007869 ms


E0828 14:03:41.923437  114032 xtile_compiler.cc:399] Fusion: gemm_fusion_dot.3 = f32[8900,1]{1,0} fusion(constant.19, bitcast.13), kind=kCustom, calls=gemm_fusion_dot.3_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["128","8"]}],"num_ctas":1,"num_stages":3,"is_tma_allowed":true,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0828 14:03:41.923501  114032 xtile_compiler.cc:401] Computation: gemm_fusion_dot.3_computation.clone {
  parameter_0 = f32[8900,3000]{1,0} parameter(0)
  parameter_1 = f32[3000,1]{1,0} parameter(1)
  ROOT dot.4 = f32[8900,1]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["64"]}
}
E0828 14:03:41.937428  114043 xtile_compiler.cc:399] Fusion: gemm

Width: 3000, Depth: 2 -> 35.724 Mparams, 0.246515 +/- 0.002531 ms
Width: 3000, Depth: 3 -> 44.727 Mparams, 0.293561 +/- 0.004523 ms
Width: 3000, Depth: 4 -> 53.730 Mparams, 0.340387 +/- 0.004740 ms


E0828 14:03:57.681479  114031 xtile_compiler.cc:399] Fusion: gemm_fusion_dot.4 = f32[8900,1]{1,0} fusion(constant.19, bitcast.21), kind=kCustom, calls=gemm_fusion_dot.4_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["128","8"]}],"num_ctas":1,"num_stages":3,"is_tma_allowed":true,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0828 14:03:57.681540  114031 xtile_compiler.cc:401] Computation: gemm_fusion_dot.4_computation.clone {
  parameter_0.1 = f32[8900,5000]{1,0} parameter(0)
  parameter_1.1 = f32[5000,1]{1,0} parameter(1)
  ROOT dot.6 = f32[8900,1]{1,0} dot(parameter_0.1, parameter_1.1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["64"]}
}
E0828 14:03:57.694527  114029 xtile_compiler.cc:399] Fusi

Width: 5000, Depth: 2 -> 69.534 Mparams, 0.417119 +/- 0.004124 ms
Width: 5000, Depth: 3 -> 94.539 Mparams, 0.559921 +/- 0.010927 ms
Width: 5000, Depth: 4 -> 119.544 Mparams, 0.675928 +/- 0.004495 ms


So even a BigPayne-eque emulator fits in our compute margin.